# Interactive Land Value Maps by Neighborhood - DEBUG VERSION

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import folium
import plotly.graph_objects as go
from pyproj import Transformer
import pyreadr
import os

In [2]:
# Load data
result = pyreadr.read_r('/Volumes/ssd_externo/UEL MESTRADO 2020/Dissertação/Simulações em R/GAMLSS/pred_geral')
df = result[None]
print(f"Total de pontos: {len(df)}")
print(f"Colunas: {list(df.columns)[:10]}...")

Total de pontos: 89089
Colunas: ['x', 'y', 'jan.2000', 'jul.2000', 'jan.2001', 'jul.2001', 'jan.2002', 'jul.2002', 'jan.2003', 'jul.2003']...


In [3]:
# Load neighborhoods
shp_path = '/Users/fjcosta/Documents/landCoverlandValue/bairros/BairrosLondrina.shp'
neighborhoods = gpd.read_file(shp_path)
neighborhoods_29192 = neighborhoods.to_crs(epsg=29192)
print(f"Bairros carregados: {len(neighborhoods_29192)}")

Bairros carregados: 66


In [4]:
# Create points GeoDataFrame
geometry = [Point(xy) for xy in zip(df['x'], df['y'])]
gdf_points = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:29192')
print(f"GeoDataFrame criado com {len(gdf_points)} pontos")

GeoDataFrame criado com 89089 pontos


In [5]:
# Spatial join
points_with_neighborhood = gpd.sjoin(
    gdf_points, 
    neighborhoods_29192[['BAIRRO', 'geometry']], 
    how='left', 
    predicate='within'
)
print(f"Pontos com bairro: {points_with_neighborhood['BAIRRO'].notna().sum()}")
print(f"Pontos sem bairro: {points_with_neighborhood['BAIRRO'].isna().sum()}")

Pontos com bairro: 79198
Pontos sem bairro: 9892


In [6]:
# Get temporal columns
temporal_cols = [col for col in df.columns if col.startswith(('jan.', 'jul.'))]
print(f"Colunas temporais: {len(temporal_cols)}")
print(f"De {temporal_cols[0]} até {temporal_cols[-1]}")

Colunas temporais: 44
De jan.2000 até jul.2021


In [7]:
# Calculate neighborhood averages
neighborhood_temporal_data = {}

for bairro in neighborhoods_29192['BAIRRO'].unique():
    points_in_bairro = points_with_neighborhood[points_with_neighborhood['BAIRRO'] == bairro]
    
    if len(points_in_bairro) > 0:
        mean_values = points_in_bairro[temporal_cols].mean()
        neighborhood_temporal_data[bairro] = mean_values.to_dict()
    else:
        neighborhood_temporal_data[bairro] = None

with_data = sum(1 for v in neighborhood_temporal_data.values() if v is not None)
without_data = sum(1 for v in neighborhood_temporal_data.values() if v is None)
print(f"Com dados: {with_data}")
print(f"Sem dados: {without_data}")

Com dados: 66
Sem dados: 0


In [8]:
# Function to create Plotly chart
def create_time_series_chart(bairro_name, temporal_data):
    if temporal_data is None:
        return "<div style='padding:20px; text-align:center; font-family:Arial;'><b>Sem dados disponíveis</b></div>"
    
    periods = list(temporal_data.keys())
    values = list(temporal_data.values())
    
    period_labels = []
    for period in periods:
        month, year = period.split('.')
        month_abbr = 'Jan' if month == 'jan' else 'Jul'
        period_labels.append(f"{month_abbr}/{year}")
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=period_labels,
        y=values,
        mode='lines+markers',
        name='Valor Médio',
        line=dict(color='#1f77b4', width=2),
        marker=dict(size=4)
    ))
    
    fig.update_layout(
        title=dict(
            text=f"<b>{bairro_name}</b>",
            x=0.5,
            xanchor='center',
            font=dict(size=16)
        ),
        xaxis_title='Período',
        yaxis_title='Valor Médio (R$/m²)',
        hovermode='x unified',
        width=650,
        height=400,
        margin=dict(l=60, r=40, t=60, b=100),
        font=dict(size=11),
        plot_bgcolor='white',
        paper_bgcolor='white',
        xaxis=dict(
            showgrid=True,
            gridcolor='lightgray',
            tickangle=-45
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor='lightgray'
        )
    )
    
    chart_html = fig.to_html(
        include_plotlyjs='cdn',
        config={'displayModeBar': False},
        div_id=None
    )
    
    # Add explanatory text below chart
    info_text = """
    <div style='padding: 15px; font-family: Arial, sans-serif; font-size: 12px; 
                background-color: #f8f9fa; border-top: 2px solid #dee2e6; margin-top: 10px;'>
        <p style='margin: 0 0 8px 0; font-weight: bold; color: #495057;'>Estimação sob a seguinte situação paradigma:</p>
        <ul style='margin: 0; padding-left: 20px; color: #6c757d; line-height: 1.6;'>
            <li>Lote plano</li>
            <li>Área de 377 m²</li>
            <li>Rua asfaltada</li>
            <li>Implantação não condominial</li>
        </ul>
    </div>
    """
    
    full_html = chart_html + info_text
    
    return full_html

In [9]:
# Generate all charts
neighborhood_charts = {}
for bairro, data in neighborhood_temporal_data.items():
    chart_html = create_time_series_chart(bairro, data)
    neighborhood_charts[bairro] = chart_html

print(f"Gráficos gerados: {len(neighborhood_charts)}")

Gráficos gerados: 66


In [10]:
# TEST: Create simple map first (without charts)
neighborhoods_latlon = neighborhoods_29192.to_crs(epsg=4326)
center_lat = neighborhoods_latlon.geometry.centroid.y.mean()
center_lon = neighborhoods_latlon.geometry.centroid.x.mean()

print(f"Centro do mapa: ({center_lat:.4f}, {center_lon:.4f})")

# Create base map
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=12,
    tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
    attr='Google Satellite'
)

print("Mapa base criado")

Centro do mapa: (-23.3094, -51.1607)
Mapa base criado


/var/folders/5z/5t77xq511rx6jf39mc48mzx40000gn/T/ipykernel_70948/558468549.py:3: UserWarning:

Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.


/var/folders/5z/5t77xq511rx6jf39mc48mzx40000gn/T/ipykernel_70948/558468549.py:4: UserWarning:

Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.




In [11]:
# Add neighborhoods as simple GeoJson (test)
folium.GeoJson(
    neighborhoods_latlon,
    style_function=lambda x: {
        'fillColor': 'transparent',
        'color': '#ffffff',
        'weight': 2,
        'fillOpacity': 0.1
    },
    tooltip=folium.GeoJsonTooltip(fields=['BAIRRO'], aliases=['Bairro:'])
).add_to(m)

print("Polígonos adicionados ao mapa")

Polígonos adicionados ao mapa


In [12]:
# Save test map (without popups)
output_dir = '/Users/fjcosta/Documents/landCoverlandValue/landvalue/temporal/'
test_path = os.path.join(output_dir, 'test_map_only.html')

m.save(test_path)
print(f"Mapa de teste salvo: {test_path}")
print("ABRA ESTE ARQUIVO PARA VERIFICAR SE O MAPA APARECE")

Mapa de teste salvo: /Users/fjcosta/Documents/landCoverlandValue/landvalue/temporal/test_map_only.html
ABRA ESTE ARQUIVO PARA VERIFICAR SE O MAPA APARECE


In [13]:
# Now create map WITH popups
m2 = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=12,
    tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
    attr='Google Satellite'
)

# Add neighborhoods with popups
for idx, row in neighborhoods_latlon.iterrows():
    bairro_name = row['BAIRRO']
    chart_html = neighborhood_charts.get(bairro_name, "<div>Erro</div>")
    
    # Create IFrame for popup (increased height for explanatory text)
    iframe = folium.IFrame(html=chart_html, width=700, height=550)
    popup = folium.Popup(iframe, max_width=700)
    
    folium.GeoJson(
        row.geometry,
        style_function=lambda x: {
            'fillColor': 'transparent',
            'color': '#ffffff',
            'weight': 1.5,
            'fillOpacity': 0.1
        },
        highlight_function=lambda x: {
            'fillColor': '#ffff00',
            'fillOpacity': 0.3,
            'weight': 3
        },
        tooltip=folium.Tooltip(bairro_name),
        popup=popup
    ).add_to(m2)

print(f"Adicionados {len(neighborhoods_latlon)} bairros com popups")

Adicionados 66 bairros com popups


In [14]:
# Save final map
final_path = os.path.join(output_dir, 'neighborhood_temporal_analysis.html')
m2.save(final_path)

file_size_mb = os.path.getsize(final_path) / (1024 * 1024)
print(f"\n{'='*60}")
print(f"Mapa final salvo: {final_path}")
print(f"Tamanho: {file_size_mb:.2f} MB")
print(f"{'='*60}")


Mapa final salvo: /Users/fjcosta/Documents/landCoverlandValue/landvalue/temporal/neighborhood_temporal_analysis.html
Tamanho: 1.47 MB
